In [1]:
# importing libraries and packages

import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV,cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import os
import json
import joblib

In [2]:
# load dataset from the (preprocessing and training data development) saved processed datasets

X_train = pd.read_csv('X_train_processed.csv')
X_test = pd.read_csv('X_test_processed.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')     

In [3]:
# Datatype & shape check

print(X_train.dtypes.value_counts())
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print(X_train.columns)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

bool       105
float64     51
object       1
Name: count, dtype: int64
X_train shape: (79474, 157)
y_train shape: (79474, 1)
Index(['time_in_hospital', 'num_lab_procedures', 'age', 'number_inpatient',
       'number_emergency', 'number_outpatient', 'num_medications',
       'num_procedures', 'number_diagnoses', 'age_mid',
       ...
       'glimepiride_dose_change_1', 'glipizide_dose_change_1',
       'glyburide_dose_change_1', 'pioglitazone_dose_change_1',
       'rosiglitazone_dose_change_1', 'acarbose_dose_change_1',
       'miglitol_dose_change_1', 'tolazamide_dose_change_1',
       'glyburide-metformin_dose_change_1', 'insulin_dose_change_1'],
      dtype='object', length=157)
X_train shape: (79474, 157)
y_train shape: (79474, 1)


In [4]:
# Drop the age column since it have been engineered as age_mid

X_train.drop(columns=['age'], inplace=True)
X_test.drop(columns=['age'], inplace=True)

# Convert target to 1D array (required by sklearn models) using .squeeze()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# Convert booleans columns to 0/1, to ensure consistency on the models
bool_col = X_train.select_dtypes(include='bool').columns

X_train[bool_col] = X_train[bool_col].astype(int)
X_test[bool_col] = X_test[bool_col].astype(int)

# Final datatype check to confirm change in datatype to int/float and shape

print(X_train.dtypes.value_counts())
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

int64      105
float64     51
Name: count, dtype: int64
X_train shape: (79474, 156)
y_train shape: (79474,)


In [5]:
# Review sample of the entire columns without truncation

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)   # prevents line wrapping
pd.set_option('display.max_colwidth', None)

X_train.sample(5)

,time_in_hospital,num_lab_procedures,number_inpatient,number_emergency,number_outpatient,num_medications,num_procedures,number_diagnoses,age_mid,metformin_flag,metformin_dose,repaglinide_flag,repaglinide_dose,nateglinide_flag,nateglinide_dose,chlorpropamide_flag,chlorpropamide_dose,glimepiride_flag,glimepiride_dose,acetohexamide_flag,acetohexamide_dose,glipizide_flag,glipizide_dose,glyburide_flag,glyburide_dose,tolbutamide_flag,tolbutamide_dose,pioglitazone_flag,pioglitazone_dose,rosiglitazone_flag,rosiglitazone_dose,acarbose_flag,acarbose_dose,miglitol_flag,miglitol_dose,troglitazone_flag,troglitazone_dose,tolazamide_flag,tolazamide_dose,glyburide-metformin_flag,glyburide-metformin_dose,glipizide-metformin_flag,glipizide-metformin_dose,glimepiride-pioglitazone_flag,glimepiride-pioglitazone_dose,metformin-rosiglitazone_flag,metformin-rosiglitazone_dose,metformin-pioglitazone_flag,metformin-pioglitazone_dose,insulin_flag,insulin_dose,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Nephrology,medical_specialty_Orthopedics,medical_specialty_Orthopedics-Reconstructive,medical_specialty_Other,medical_specialty_Radiologist,medical_specialty_Surgery-General,medical_specialty_Unknown,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,age_range_[10-20),age_range_[20-30),age_range_[30-40),age_range_[40-50),age_range_[50-60),age_range_[60-70),age_range_[70-80),age_range_[80-90),age_range_[90-100),admission_type_id_2,admission_type_id_3,admission_type_id_4,admission_type_id_5,admission_type_id_6,admission_type_id_7,admission_type_id_8,discharge_disposition_id_2,discharge_disposition_id_3,discharge_disposition_id_4,discharge_disposition_id_5,discharge_disposition_id_6,discharge_disposition_id_7,discharge_disposition_id_8,discharge_disposition_id_9,discharge_disposition_id_10,discharge_disposition_id_12,discharge_disposition_id_15,discharge_disposition_id_16,discharge_disposition_id_17,discharge_disposition_id_18,discharge_disposition_id_22,discharge_disposition_id_23,discharge_disposition_id_24,discharge_disposition_id_25,discharge_disposition_id_27,discharge_disposition_id_28,admission_source_id_2,admission_source_id_3,admission_source_id_4,admission_source_id_5,admission_source_id_6,admission_source_id_7,admission_source_id_8,admission_source_id_9,admission_source_id_10,admission_source_id_11,admission_source_id_13,admission_source_id_14,admission_source_id_17,admission_source_id_20,admission_source_id_22,admission_source_id_25,change_No,diabetesMed_Yes,diag_1_3digit_276,diag_1_3digit_38,diag_1_3digit_410,diag_1_3digit_414,diag_1_3digit_427,diag_1_3digit_428,diag_1_3digit_434,diag_1_3digit_486,diag_1_3digit_491,diag_1_3digit_493,diag_1_3digit_584,diag_1_3digit_599,diag_1_3digit_682,diag_1_3digit_715,diag_1_3digit_780,diag_1_3digit_786,diag_1_3digit_820,diag_1_3digit_996,diag_1_3digit_Other,diag_1_3digit_V57,metformin_dose_change_1,repaglinide_dose_change_1,nateglinide_dose_change_1,chlorpropamide_dose_change_1,glimepiride_dose_change_1,glipizide_dose_change_1,glyburide_dose_change_1,pioglitazone_dose_change_1,rosiglitazone_dose_change_1,acarbose_dose_change_1,miglitol_dose_change_1,tolazamide_dose_change_1,glyburide-metformin_dose_change_1,insulin_dose_change_1
12062,-0.801850,1.330119,-0.498835,-0.207036,-0.289622,-0.493141,-0.784188,0.305637,1.207384,-0.500869,-0.464976,-0.124549,-0.118014,-0.084088,-0.080589,-0.029478,-0.028317,-0.233318,-0.216946,-0.003547,-0.003547,-0.380303,-0.349373,-0.343508,-0.313204,-0.014627,-0.014627,-0.281336,-0.269881,-0.261397,-0.252221,-0.055608,-0.05375,-0.020381,-0.01807,-0.005017,-0.005017,-0.019754,-0.019471,-0.08424,-0.082168,-0.011218,-0.011218,-0.003547,-0.003547,-0.005017,-0.005017,-0.003547,-0.003547,0.934779,0.114251,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,

## Goals / Objective (Problem Statement)
The aim of this project is to explore how machine learning can predict hospital readmission risk and uncover key patient and clinical factors. The model prioritizes identifying high-risk patients (high recall), enabling hospitals to reduce avoidable readmissions, improve patient outcomes, and manage healthcare costs more effectively.

🎯 Key decision:

Primary metric → Recall

Secondary → Precision (to control false alarms)/ F1

## DATA UNDERSTANDING OF THE FEATURES

Key feature groups in this data:

#### Utilization:

number_inpatient, number_emergency, number_outpatient
Severity
time_in_hospital, number_diagnoses


#### Medication:

*_flag, *_dose, *_dose_change_1


#### Demographics:

age_mid, race_*, gender_*


#### Admission / discharge:

admission_type_id_*, discharge_disposition_id_*


#### Diagnosis groups:

diag_1_3digit_*

In [6]:
# Understanding the data

X_train.info()
X_train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79474 entries, 0 to 79473
Columns: 156 entries, time_in_hospital to insulin_dose_change_1
dtypes: float64(51), int64(105)
memory usage: 94.6 MB


,time_in_hospital,num_lab_procedures,number_inpatient,number_emergency,number_outpatient,num_medications,num_procedures,number_diagnoses,age_mid,metformin_flag,metformin_dose,repaglinide_flag,repaglinide_dose,nateglinide_flag,nateglinide_dose,chlorpropamide_flag,chlorpropamide_dose,glimepiride_flag,glimepiride_dose,acetohexamide_flag,acetohexamide_dose,glipizide_flag,glipizide_dose,glyburide_flag,glyburide_dose,tolbutamide_flag,tolbutamide_dose,pioglitazone_flag,pioglitazone_dose,rosiglitazone_flag,rosiglitazone_dose,acarbose_flag,acarbose_dose,miglitol_flag,miglitol_dose,troglitazone_flag,troglitazone_dose,tolazamide_flag,tolazamide_dose,glyburide-metformin_flag,glyburide-metformin_dose,glipizide-metformin_flag,glipizide-metformin_dose,glimepiride-pioglitazone_flag,glimepiride-pioglitazone_dose,metformin-rosiglitazone_flag,metformin-rosiglitazone_dose,metformin-pioglitazone_flag,metformin-pioglitazone_dose,insulin_flag,insulin_dose,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Nephrology,medical_specialty_Orthopedics,medical_specialty_Orthopedics-Reconstructive,medical_specialty_Other,medical_specialty_Radiologist,medical_specialty_Surgery-General,medical_specialty_Unknown,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,age_range_[10-20),age_range_[20-30),age_range_[30-40),age_range_[40-50),age_range_[50-60),age_range_[60-70),age_range_[70-80),age_range_[80-90),age_range_[90-100),admission_type_id_2,admission_type_id_3,admission_type_id_4,admission_type_id_5,admission_type_id_6,admission_type_id_7,admission_type_id_8,discharge_disposition_id_2,discharge_disposition_id_3,discharge_disposition_id_4,discharge_disposition_id_5,discharge_disposition_id_6,discharge_disposition_id_7,discharge_disposition_id_8,discharge_disposition_id_9,discharge_disposition_id_10,discharge_disposition_id_12,discharge_disposition_id_15,discharge_disposition_id_16,discharge_disposition_id_17,discharge_disposition_id_18,discharge_disposition_id_22,discharge_disposition_id_23,discharge_disposition_id_24,discharge_disposition_id_25,discharge_disposition_id_27,discharge_disposition_id_28,admission_source_id_2,admission_source_id_3,admission_source_id_4,admission_source_id_5,admission_source_id_6,admission_source_id_7,admission_source_id_8,admission_source_id_9,admission_source_id_10,admission_source_id_11,admission_source_id_13,admission_source_id_14,admission_source_id_17,admission_source_id_20,admission_source_id_22,admission_source_id_25,change_No,diabetesMed_Yes,diag_1_3digit_276,diag_1_3digit_38,diag_1_3digit_410,diag_1_3digit_414,diag_1_3digit_427,diag_1_3digit_428,diag_1_3digit_434,diag_1_3digit_486,diag_1_3digit_491,diag_1_3digit_493,diag_1_3digit_584,diag_1_3digit_599,diag_1_3digit_682,diag_1_3digit_715,diag_1_3digit_780,diag_1_3digit_786,diag_1_3digit_820,diag_1_3digit_996,diag_1_3digit_Other,diag_1_3digit_V57,metformin_dose_change_1,repaglinide_dose_change_1,nateglinide_dose_change_1,chlorpropamide_dose_change_1,glimepiride_dose_change_1,glipizide_dose_change_1,glyburide_dose_change_1,pioglitazone_dose_change_1,rosiglitazone_dose_change_1,acarbose_dose_change_1,miglitol_dose_change_1,tolazamide_dose_change_1,glyburide-metformin_dose_change_1,insulin_dose_change_1
count,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.947400e+04,7.9474

In [7]:
# Discharge-related variables were removed to prevent data leakage, as they may encode post-outcome information unavailable at prediction time.

leakage_cols = [col for col in X_train.columns if "discharge_disposition" in col]

X_train = X_train.drop(columns=leakage_cols)
X_test = X_test.drop(columns=leakage_cols)

In [8]:
# Ensure all column names are strings which address XGBoost Model concerns
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

# Clean invalid characters
X_train.columns = (
    X_train.columns
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
    .str.replace('<', 'lt_', regex=False)
)

X_test.columns = (
    X_test.columns
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
    .str.replace('<', 'lt_', regex=False)
)

In [9]:
# Additional Feature Engineering - improve model performance due to weak signals

def add_features(df):
    df = df.copy()
    
    df['total_visits'] = (
        df['number_inpatient'] +
        df['number_emergency'] +
        df['number_outpatient']
    )
    
    df['high_utilizer'] = (df['total_visits'] > 2).astype(int)
    
    df['medication_count'] = df[[col for col in df.columns if "_flag" in col]].sum(axis=1)
    
    return df

X_train = add_features(X_train)
X_test = add_features(X_test)


# Scaling of our continuous variable dataset
num_cols = X_train.select_dtypes(include=['float64']).columns

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train.dtypes.value_counts())

print(list(X_train.columns))

int64      86
float64    53
Name: count, dtype: int64
['time_in_hospital', 'num_lab_procedures', 'number_inpatient', 'number_emergency', 'number_outpatient', 'num_medications', 'num_procedures', 'number_diagnoses', 'age_mid', 'metformin_flag', 'metformin_dose', 'repaglinide_flag', 'repaglinide_dose', 'nateglinide_flag', 'nateglinide_dose', 'chlorpropamide_flag', 'chlorpropamide_dose', 'glimepiride_flag', 'glimepiride_dose', 'acetohexamide_flag', 'acetohexamide_dose', 'glipizide_flag', 'glipizide_dose', 'glyburide_flag', 'glyburide_dose', 'tolbutamide_flag', 'tolbutamide_dose', 'pioglitazone_flag', 'pioglitazone_dose', 'rosiglitazone_flag', 'rosiglitazone_dose', 'acarbose_flag', 'acarbose_dose', 'miglitol_flag', 'miglitol_dose', 'troglitazone_flag', 'troglitazone_dose', 'tolazamide_flag', 'tolazamide_dose', 'glyburide-metformin_flag', 'glyburide-metformin_dose', 'glipizide-metformin_flag', 'glipizide-metformin_dose', 'glimepiride-pioglitazone_flag', 'glimepiride-pioglitazone_dose', 'met

In [10]:
# Additional Feature Engineering (Due 

# Severity Interactions (High impact) - Captures how serious those visits are
X_train['severity_index'] = X_train['time_in_hospital'] * X_train['number_diagnoses']
X_test['severity_index'] = X_test['time_in_hospital'] * X_test['number_diagnoses']

X_train['visit_severity'] = X_train['total_visits'] * X_train['time_in_hospital']
X_test['visit_severity'] = X_test['total_visits'] * X_test['time_in_hospital']

# Utilization (weighted) - Inpatient visits are more serious → weight them higher
X_train['utilization_intensity'] = (
    X_train['number_inpatient'] * 3 +
    X_train['number_emergency'] * 2 +
    X_train['number_outpatient']
)

X_test['utilization_intensity'] = (
    X_test['number_inpatient'] * 3 +
    X_test['number_emergency'] * 2 +
    X_test['number_outpatient']
)

# Medical Pressure - High meds in short stay = unstable patient
X_train['medication_pressure'] = X_train['num_medications'] / (X_train['time_in_hospital'] + 1)
X_test['medication_pressure'] = X_test['num_medications'] / (X_test['time_in_hospital'] + 1)


# Diagnosis Burden
diag_cols = [col for col in X_train.columns if "diag_1_3digit" in col]

X_train['diag_burden'] = X_train[diag_cols].sum(axis=1)
X_test['diag_burden'] = X_test[diag_cols].sum(axis=1)

# Risk Flag (Non-Linear Signal)
X_train['critical_patient'] = (
    (X_train['number_inpatient'] > 2) |
    (X_train['time_in_hospital'] > 7) |
    (X_train['num_medications'] > 15)
).astype(int)

X_test['critical_patient'] = (
    (X_test['number_inpatient'] > 2) |
    (X_test['time_in_hospital'] > 7) |
    (X_test['num_medications'] > 15)
).astype(int)

In [11]:
# Feature Selection (Noise reduction) - Due to High cardinality

# Reduce noise (e.g., 157 → 80 features)
selector = SelectKBest(score_func=f_classif, k=80)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Restore column names (IMPORTANT)
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask]

X_train_selected = pd.DataFrame(X_train_selected, columns=selected_features)
X_test_selected = pd.DataFrame(X_test_selected, columns=selected_features)

print("New shape:", X_train_selected.shape)

New shape: (79474, 80)


In [12]:
# Handle class imbalance - Compute imbalance rati
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

print("Scale pos weight:", scale_pos_weight)

Scale pos weight: 7.780687216882113


In [13]:
# Model 1: Logistic Regression (Baseline and Interpretable)

log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train_selected, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [14]:
# Model 2: Random Forest (Robust Non-Linear)
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [15]:
# Model 3: Gradient Boosting (Performane Model)

gb = GradientBoostingClassifier()
gb.fit(X_train_selected, y_train)

GradientBoostingClassifier()

In [16]:
# Model 4: XGBooost

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.5,
    reg_lambda=1.5,
    scale_pos_weight=scale_pos_weight,  # imbalance handled here
    random_state=42,
    n_jobs=-1,
    eval_metric='auc'
)

xgb.fit(X_train_selected, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [17]:
# Models validation

models = {
    "Logistic Regression": log_model,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    "XGB": xgb
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_selected,
        y_train,
        cv=3,
        scoring='roc_auc'
    )
    
    print(f"{name} ROC-AUC: {scores.mean():.3f}")

Logistic Regression ROC-AUC: 0.644
Random Forest ROC-AUC: 0.613
Gradient Boosting ROC-AUC: 0.646
XGB ROC-AUC: 0.642


Although Gradient Boosting achieved the highest ROC-AUC (0.646), XGBoost (0.642) was selected as the final model due to its robustness, regularization capabilities, and ability to capture nonlinear relationships and feature interactions. The difference in ROC-AUC across models was minimal, indicating comparable performance. XGBoost was therefore preferred for its stability and flexibility, particularly in combination with threshold tuning to optimize recall for the healthcare objective of minimizing missed readmissions.

In [18]:
# Hyperparameter tuning based on the best selected model XGB

# Refined XGBoost Hyperparameter

param_dist_xgb = {
    'n_estimators': randint(200, 500),        # reduced range
    'max_depth': randint(3, 6),               
    'learning_rate': uniform(0.02, 0.05),     
    'subsample': uniform(0.75, 0.2),          
    'colsample_bytree': uniform(0.75, 0.2)    
}

xgb_search = RandomizedSearchCV(
    xgb,
    param_dist_xgb,
    n_iter=10,          #  cut from 25 → 10
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train_selected, y_train)

best_xgb = xgb_search.best_estimator_

print("Best XGBoost Params:", xgb_search.best_params_)
print("Best CV Recall:", xgb_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best XGBoost Params: {'colsample_bytree': np.float64(0.9116794696232923), 'learning_rate': np.float64(0.03523068845866854), 'max_depth': 3, 'n_estimators': 291, 'subsample': np.float64(0.8380304987479202)}
Best CV Recall: 0.6474347538591925


In [19]:
# Probability predictions for the RF & GB models

y_probs_xgb = best_xgb.predict_proba(X_test_selected)[:, 1]

thresholds = np.arange(0.05, 0.5, 0.01)

results = []

roc_auc = roc_auc_score(y_test, y_probs_xgb)

for t in thresholds:
    preds = (y_probs_xgb >= t).astype(int)
    
    results.append({
        "threshold": t,
        "recall": recall_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc
    })

results_df = pd.DataFrame(results)

In [20]:
pd.set_option('display.max_rows', None)

print(results_df)

    threshold    recall  precision        f1   roc_auc
0        0.05  1.000000   0.113896  0.204500  0.649317
1        0.06  1.000000   0.113896  0.204500  0.649317
2        0.07  1.000000   0.113896  0.204500  0.649317
3        0.08  1.000000   0.113896  0.204500  0.649317
4        0.09  1.000000   0.113896  0.204500  0.649317
5        0.10  1.000000   0.113896  0.204500  0.649317
6        0.11  1.000000   0.113896  0.204500  0.649317
7        0.12  1.000000   0.113896  0.204500  0.649317
8        0.13  1.000000   0.113896  0.204500  0.649317
9        0.14  1.000000   0.113902  0.204510  0.649317
10       0.15  1.000000   0.113930  0.204556  0.649317
11       0.16  0.999558   0.113932  0.204549  0.649317
12       0.17  0.999558   0.113989  0.204641  0.649317
13       0.18  0.999558   0.114150  0.204901  0.649317
14       0.19  0.999116   0.114255  0.205061  0.649317
15       0.20  0.999116   0.114423  0.205331  0.649317
16       0.21  0.999116   0.114644  0.205686  0.649317
17       0

In [21]:
# Final Model & Threshold  Selection

# Example constraint-based selection
candidate = results_df[
    (results_df["recall"] >= 0.70)
]

candidate = candidate.sort_values(by="precision", ascending=False)

final_row = candidate.iloc[0]

final_model = best_xgb
final_threshold = final_row["threshold"]
final_probs = y_probs_xgb

print("Selected threshold:", final_threshold)

Selected threshold: 0.45000000000000007


A threshold of 0.45 was selected as it provided the best balance between recall (≈0.71) and precision (≈0.15). This ensures that a majority of high-risk patients are identified while maintaining a manageable false positive rate.

In [22]:
# Final Model evaluation

y_pred_final = (final_probs >= final_threshold).astype(int)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_final))
print("\nClassification Report:\n", classification_report(y_test, y_pred_final))

Confusion Matrix:
 [[8547 9059]
 [ 660 1603]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.49      0.64     17606
           1       0.15      0.71      0.25      2263

    accuracy                           0.51     19869
   macro avg       0.54      0.60      0.44     19869
weighted avg       0.84      0.51      0.59     19869



The model prioritizes recall to minimize missed high-risk patients, which is critical in healthcare settings where undetected readmission risk can lead to severe patient outcomes and increased costs.

In [23]:
# Feature Importance

importances = pd.Series(
    final_model.feature_importances_,
    index=selected_features
).sort_values(ascending=False)

print(importances.head(15))

number_inpatient         0.140565
utilization_intensity    0.065762
insulin_dose_change_1    0.033440
age_mid                  0.028777
visit_severity           0.024663
diag_1_3digit_434        0.024517
number_diagnoses         0.023178
time_in_hospital         0.023079
diag_1_3digit_428        0.018828
insulin_dose             0.017537
diabetesMed_Yes          0.017130
diag_1_3digit_786        0.014584
diag_1_3digit_820        0.014548
critical_patient         0.013977
metformin_flag           0.013212
dtype: float32


📊 Discussion

This study developed a machine learning model to predict hospital readmissions using a refined pipeline combining feature engineering, feature selection, imbalance-aware learning, and threshold optimization. Model performance improved from ROC-AUC ~0.61 to ~0.65, indicating that feature quality, not model complexity, was the main driver of improvement.

Feature importance analysis revealed that number_inpatient was the strongest predictor, confirming that prior hospital utilization is the most reliable indicator of future readmission. Engineered features such as utilization_intensity and visit_severity ranked highly, demonstrating that combining variables into clinically meaningful measures significantly enhances predictive signal.

Medication-related features, including insulin dose changes and diabetes medication usage, also contributed strongly, suggesting that treatment adjustments reflect patient instability. Additional factors such as age, number of diagnoses, and length of stay further captured patient complexity, while key diagnosis groups highlighted condition-specific risks. The inclusion of the critical_patient feature confirms that engineered non-linear risk indicators improve model effectiveness.

Threshold optimization was essential in aligning the model with clinical priorities. A threshold of appox. 0.45 achieved appox. 71% recall, ensuring most high-risk patients are identified. Although precision (~15%) remains low, this trade-off is acceptable in healthcare settings where missing high-risk patients is more critical than over-flagging. The model is therefore best suited as a screening tool for early intervention.

⚠️ Limitations

Limited data signal: Key factors such as socio-economic status, post-discharge care, and patient history are not included, constraining performance.
Low precision: High false positives may reduce operational efficiency.
Static features: Lack of temporal data limits the model’s ability to capture patient progression.
Generalizability: Results may not transfer across different healthcare settings.
Threshold sensitivity: Performance depends on the chosen operating threshold.

🎯 Conclusion Insight

The project shows that effective readmission prediction depends on capturing patient severity and utilization patterns, not just model choice. The final model provides a recall-focused, clinically useful screening tool, but should be used alongside broader decision-support systems.